# 05 — SARSA(λ) debugging, curriculum training, and Tasks 1–4

This notebook records the reproducible short-run experiment executed on 2026-09-11. The agent uses the same 17 action-relative features and safe-action mask as `linear_q_agent`, with accumulating traces and true on-policy next actions. Evaluation uses unseen seeds and `epsilon=0`.

## Algorithm and debugging checks

$Q(s,a)=w^T\phi(s,a)$, $\delta=r+\gamma Q(s',a')-Q(s,a)$, $e\leftarrow\gamma\lambda e+\phi(s,a)$, $w\leftarrow w+\alpha\delta e$.

Parameters: `gamma=0.95`, `lambda=0.70`, `alpha=0.003`, accumulating traces, TD clipping at 20, epsilon 1.0→0.03. The callback caches the transition and waits for the next `act` call, so $a'$ is the action actually selected by the behavior policy. Terminal updates do not bootstrap and traces are reset after every round. Eight focused feature/model/SARSA tests passed; a 3-round framework smoke training run completed without exceptions.

In [ ]:
# Run from the repository root to reproduce the checks.
import subprocess
subprocess.run(['.venv/bin/python', '-m', 'unittest',
                'tests.test_sarsa_lambda_agent', 'tests.test_linear_q_agent', '-v'], check=True)

## Executed curriculum

| Task | Training (seed) | Held-out evaluation (seed) |
|---|---|---|
| 1: coin navigation | 30 coin-heaven rounds (51101) | 10 rounds (51191) |
| 2: crates/bombs/escape | 20 classic solo rounds (51201) | 10 rounds (51291) |
| 3: opponent interaction | 20 classic rounds vs 3 peaceful agents (51301) | 10 rounds (51391) |
| 4: tournament pressure | no evaluation-time learning | 10 classic rounds vs 3 rule agents (51491) |

Raw framework statistics are stored in `results/sarsa_task{1..4}_eval.json`; training statistics are stored for Tasks 1–3.

In [1]:
# Values below are recomputed from the saved raw framework results.
from pathlib import Path
from evaluation.summarize_stats import summarize, summarize_match
for task in range(1, 5):
    path = Path(f'results/sarsa_task{task}_eval.json')
    print(task, summarize([path], 'sarsa-lambda'))
    print(summarize_match([path]))

Task  mean_score  coins  crates  kills  suicides  invalid  survival  win_rate
1     50.00       50.00  0.00    0.00   0.00      0.00     1.00      n/a
2      2.20        2.20 34.40    0.00   0.00      0.00     1.00      n/a
3      2.40        0.90 23.30    0.30   0.30      0.10     0.70      1.00
4      2.30        1.80 11.50    0.10   0.90      1.80     0.10      0.20


## Interpretation

Task 1 is solved in this held-out sample (50/50 coins in every round). Task 2 shows safe bomb-and-crate behavior: 34.4 crates and 2.2 coins per round, with no suicide or invalid action. Task 3 produces opponent kills and wins all ten matches against passive opponents, but the 0.30 suicide rate reveals incomplete bomb safety. Task 4 is the clear stress failure: only 20% win rate, 90% suicide rate, and 2.3 mean score against rule agents. Thus the implementation is a valid, learning baseline, but this 73-round curriculum is not a converged Task-4 competitor.

Final checkpoint diagnostics: 73 episodes, 28,995 updates, finite weights, $||w||_2=2.2601$. For a report-quality comparison, train several thousand rounds, select checkpoints on separate validation seeds, and rerun a paired 100+ round suite against the frozen DDQN. The repository's archived DDQN result is useful context but is not treated here as a paired comparison because its lineups/seeds differ.

## Formal continuation: λ sweep and checkpoint selection (2026-09-11)

Starting from the common 73-episode checkpoint, three branches were trained for 40 identical Task-4 rounds (seed 52001) with `lambda ∈ {0.4, 0.7, 0.9}`, `alpha=0.003`, epsilon scheduled from 0.15 toward 0.01, and a new bomb-escape shaping term based on simulated post-bomb survival breadth. Evaluation used 20 unseen rounds (seed 52101), with no learning and epsilon fixed at zero.

In [2]:
# Recompute the lambda sweep table from raw validation files.
for suffix in ('04', '07', '09'):
    path = Path(f'results/sarsa_l{suffix}_validation.json')
    name = f'sarsa_lambda_l{suffix}'
    print(next(row for row in summarize_match([path]) if row['agent'] == name))

lambda score win_rate suicides survival kills crates
0.4    3.80  0.55     0.25     0.70     0.05  28.45
0.7    1.15  0.15     0.50     0.50     0.05   6.50
0.9    3.70  0.35     0.60     0.25     0.25  18.05


The validation result rejects the previous default λ=0.7. λ=0.4 has the highest score and win rate and the lowest suicide rate, so its 113-episode checkpoint was frozen as `frozen_sarsa_l04_stage1`. All three checkpoints remained finite; weight norms were 1.864, 1.978, and 3.322 for λ=0.4, 0.7, and 0.9 respectively.

## Low-alpha refinement and final selection

The selected λ=0.4 branch continued for 40 more Task-4 rounds (seed 52201) with `alpha=0.001` and epsilon rescheduled from 0.05 toward 0.01. Both the pre- and post-refinement checkpoints were then compared on the same new 20-round selection seed 52301.

| checkpoint | episodes | score | win rate | suicides | survival | bombs |
|---|---:|---:|---:|---:|---:|---:|
| stage 1 | 113 | 3.50 | 0.50 | 0.50 | 0.50 | 36.75 |
| stage 2 | 153 | **3.80** | 0.45 | **0.35** | **0.65** | 20.65 |

Stage 2 is promoted as `frozen_sarsa_best`: it trades 0.05 absolute win rate for higher mean score, fewer suicides, higher survival, and substantially less bomb spam. This choice follows the declared score-plus-safety criterion and demonstrates why the last checkpoint must be validated rather than automatically accepted. Total continuation training was 160 rounds across the sweep and refinement, in addition to the initial 73-round curriculum.